In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val
from utils.test import generate_test_loader, generate_test_predictions
from data.simulate_walk_the_book import simulate_walk_the_book
import warnings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

device: cpu


In [12]:
import sys, os

# Paths and volume_to_fill
root_path = "../20251020_LMU_Data_Science_Practical 2"
root = Path(root_path) if Path(root_path).exists() else Path.cwd()

import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

data_assets = [folder_name for folder_name in os.listdir(root) if "USDT" in folder_name]

paths = {asset: root / asset / "X_train.parquet" for asset in data_assets}

paths


{'BTCUSDT': PosixPath('../20251020_LMU_Data_Science_Practical 2/BTCUSDT/X_train.parquet'),
 'XRPUSDT': PosixPath('../20251020_LMU_Data_Science_Practical 2/XRPUSDT/X_train.parquet'),
 'ETHUSDT': PosixPath('../20251020_LMU_Data_Science_Practical 2/ETHUSDT/X_train.parquet'),
 'ADAUSDT': PosixPath('../20251020_LMU_Data_Science_Practical 2/ADAUSDT/X_train.parquet'),
 'SOLUSDT': PosixPath('../20251020_LMU_Data_Science_Practical 2/SOLUSDT/X_train.parquet'),
 'DOGEUSDT': PosixPath('../20251020_LMU_Data_Science_Practical 2/DOGEUSDT/X_train.parquet'),
 'LTCUSDT': PosixPath('../20251020_LMU_Data_Science_Practical 2/LTCUSDT/X_train.parquet')}

In [19]:
def infer_tick_from_spread(df):
    spread = df["ask_price_1"] - df["bid_price_1"]
    spread = spread[np.isfinite(spread)]
    spread = spread[spread > 0]

    spread = np.round(spread, 8)

    values, counts = np.unique(spread, return_counts=True)
    return values[np.argmax(counts)]

for asset, X_path in paths.items():

    df = pd.read_parquet(X_path)

    tick = infer_tick_from_spread(df)
    print(f"estimated tick size for {asset}", tick)

estimated tick size for BTCUSDT 0.1199
estimated tick size for XRPUSDT 0.0
estimated tick size for ETHUSDT 0.0
estimated tick size for ADAUSDT 0.0001498
estimated tick size for SOLUSDT 0.0
estimated tick size for DOGEUSDT 0.0
estimated tick size for LTCUSDT 0.0056


In [16]:
print(prices[:20])

[122.84643     8.792782  122.83292    10.050366  122.81941    10.447388
 122.797794   36.209636  122.776178   31.318888  122.876152   18.336716
 122.862642   16.369812  122.843728   52.821068  122.822112   15.369658
 122.8227875  59.204865 ]


In [10]:
data_assets

['BTCUSDT', 'XRPUSDT', 'ETHUSDT', 'ADAUSDT', 'SOLUSDT', 'DOGEUSDT', 'LTCUSDT']